# E4-D — Arsitektur, modif D: residual connection (mini-ResNet)
**Interpretasi dari E4-C:** menambah kapasitas malah menurunkan test acc 56.7% → **54.6%** dan waktu training naik 2.3x (67s → 152s). Masalahnya: plain deep network dengan lr kecil sulit dioptimasi — gradient harus melewati 8 layer berurutan; kapasitas ekstra tidak terpakai dalam 3 epoch.

**Modif D:** ganti blok conv biasa dengan **residual block** (shortcut `x + F(x)` ala ResNet), downsample pakai stride-2 conv:
- 3 stage (32/64/128 channel), tiap stage = conv+BN+ReLU + 1 ResBlock
- Dropout 0.3 + fc tunggal 128→10 sebagai head

Shortcut membuat gradient 'jalan tol' ke layer awal, jadi depth benar-benar terpakai. Hyperparameter tetap SGD lr=0.0025, batch 10.

## Run

In [1]:
from exp_common import *
model = ConvNetD(dropout=0.3)
res = run_experiment(
    'E4_arch_D', model,
    train_files=['data_batch_1', 'data_batch_3', 'data_batch_4', 'data_batch_5'],
    val_files=['data_batch_2'],
    epochs=3, batch_size=10, lr=0.0025, optimizer_type='sgd',
    loss_mode='ce', augment=True, fix_channels=True)


===== E4_arch_D =====
device=cuda, epochs=3, batch=10, lr=0.0025, opt=sgd, loss=ce, augment=True, fix_channels=True
params total=482,730 trainable=482,730


train=40000 val=10000


epoch 0: train_loss=1.6074 train_acc=0.4081 val_loss=1.3503 val_acc=0.5089 (68.7s, 583 samples/s)


epoch 1: train_loss=1.2174 train_acc=0.5666 val_loss=0.9489 val_acc=0.6705 (74.8s, 535 samples/s)


epoch 2: train_loss=1.0509 train_acc=0.6325 val_loss=0.9201 val_acc=0.6707 (67.5s, 593 samples/s)
peak GPU mem: 157 MB


TEST ACC = 0.6657  (eval 0.8s, train 210.9s)
saved: C:\Users\dgash\Downloads\ml_task_cnn\experiments\results\E4_arch_D.json
